# Korpus-Analyse: Semantische Ähnlichkeit, Textlängen, Bidirektionale Named Entities, Diversität (TTR/MATTR) & Quantitative Regel-Adhärenz

Dieses Notebook dient der umfassenden statistischen und linguistischen Untersuchung des erstellten Korpus aus **12 Datenquellen** mit insgesamt **830 Artikelpaaren** in Alltagssprache (AS) und Leichter Sprache (LS).

Analysiert werden:
1. **Textlängen & Kompression**: Wort- und BPE-Tokens (`tiktoken cl100k_base`), Kompressions- und Expansionsraten sowie **systematische Truncation- und Informationsverlustanalyse bei Limits von 256, 512 und 1024 Tokens**.
2. **Semantische Ähnlichkeit**: Dense Bi-Encoder Embeddings (`jinaai/jina-embeddings-v2-base-de`, 8192 Token Context).
3. **Bidirektionale Named Entities (NER)**: Erhalt von Schlüsselinformationen (Recall $AS \to LS$) vs. Vermeidung halluzinierter/hinzugedichteter Entitäten (Recall $LS \to AS$).
4. **Lexikalische Diversität**: Type-Token Ratio (TTR), Moving-Average TTR (MATTR-50) und Längen-Bias-Korrektur.
5. **Quantitative Regel-Adhärenz**: Einhaltung von 14 linguistischen LS-Regeln (Satzlänge, Passiv, Genitiv, Konjunktiv, Nominalstil, Komposita-Trennung, WSTF etc.) nach offiziellen Regelwerken.
6. **Multivariate Zusammenhänge**: Korrelationen zwischen semantischer Ähnlichkeit, Längenverhältnissen, NER und Wortschatzdiversität.

In [ ]:
import os, sys

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
import spacy
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
import tiktoken
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 11

print("Bibliotheken erfolgreich importiert.")


## 1. Datensatz laden

Wir laden die Master-CSV-Datei (`data/analysis/corpus_master.csv`), die alle vorberechneten Textpaare, semantischen Ähnlichkeiten, bidirektionalen NER-Werte sowie Lesbarkeits- und Diversitätsmetriken enthält.

In [ ]:
# Pfad zur Master-CSV-Datei
csv_candidates = [
    "../../../data/analysis/corpus_master.csv",
    "../../data/analysis/corpus_master.csv",
    "data/analysis/corpus_master.csv"
]
CSV_PATH = next((p for p in csv_candidates if os.path.exists(p)), "data/analysis/corpus_master.csv")

df = pd.read_csv(CSV_PATH)
print(f"Erfolgreich {len(df)} Artikelpaare geladen aus: {CSV_PATH}")

# Berechnung der BPE-Tokens (tiktoken cl100k_base)
enc = tiktoken.get_encoding("cl100k_base")
df["as_bpe_tokens"] = df["as_text"].fillna("").apply(lambda t: len(enc.encode(str(t))))
df["ls_bpe_tokens"] = df["ls_text"].fillna("").apply(lambda t: len(enc.encode(str(t))))

# Wort- und BPE-Längenverhältnisse
df["ratio_ls_as_word"] = df["ls_tokens"] / df["as_tokens"]
df["ratio_ls_as_bpe"] = df["ls_bpe_tokens"] / df["as_bpe_tokens"]
if "ratio_ls_as" not in df.columns:
    df["ratio_ls_as"] = df["ratio_ls_as_word"]

print(f"Spalten im DataFrame ({len(df.columns)}):", list(df.columns))
df[["source", "semantic_similarity_8192", "as_tokens", "ls_tokens", "as_bpe_tokens", "ls_bpe_tokens"]].head(3)


### 1.1 Mathematische & Algorithmische Definitionen (TTR, MATTR & Bidirektionales NER)

Für Transparenz und Reproduzierbarkeit sind hier die mathematischen und algorithmischen Definitionen hinterlegt:

- **Type-Token Ratio (TTR)**:
  $$TTR = \frac{|V|}{N}$$
  wobei $|V|$ die Anzahl der einzigartigen Lemmata/Wortformen (Types) und $N$ die Gesamtwortanzahl (Tokens) ist.

- **Moving-Average Type-Token Ratio (MATTR)**:
  Berechnung von TTR über ein gleitendes Fenster der festen Breite $W$ (Standard: $W=50$):
  $$MATTR = \frac{1}{N - W + 1} \sum_{i=1}^{N - W + 1} \frac{|V_{i, W}|}{W}$$

- **Bidirektionaler Named Entity Recall (NER)**:
  Sei $E_{AS}$ die Menge aller in der Alltagssprache extrahierten Entitäten (Personen, Orte, Organisationen) und $E_{LS}$ die Menge der Entitäten in Leichter Sprache:
  
  1. **Faktenerhalt ($AS \to LS$, Standard Recall)**:
     $$\text{Recall}_{AS \to LS} = \frac{|E_{AS} \cap E_{LS}|}{|E_{AS}|}$$
     Gibt an, welcher Anteil der im Original genannten Entitäten in der vereinfachten Version erhalten bleibt.
     
  2. **Faktentreue / Inverted Recall ($LS \to AS$)**:
     $$\text{Recall}_{LS \to AS} = \frac{|E_{LS} \cap E_{AS}|}{|E_{LS}|}$$
     Gibt an, welcher Anteil der in Leichter Sprache vorkommenden Entitäten im Ausgangstext verankert ist (Prüfung auf neue/erfundene Fakten oder Umschreibungen).

In [ ]:
def compute_ttr_and_mattr(text, window_size=50):
    """Berechnet einfache Wort-Tokens, TTR und MATTR für einen Text."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0, np.nan, np.nan
    tokens = re.findall(r'\b\w+\b', text.lower())
    n_tokens = len(tokens)
    if n_tokens == 0:
        return 0, np.nan, np.nan
    unique_types = len(set(tokens))
    ttr = unique_types / n_tokens
    if n_tokens < window_size:
        mattr = ttr
    else:
        window_ttrs = [
            len(set(tokens[i : i + window_size])) / window_size
            for i in range(n_tokens - window_size + 1)
        ]
        mattr = float(np.mean(window_ttrs))
    return n_tokens, ttr, mattr

def extract_entities_and_recall(as_text, ls_text, nlp_model=None):
    """Extrahiert Named Entities mit spaCy und berechnet den bidirektionalen Recall."""
    if nlp_model is None:
        nlp_model = spacy.load("de_core_news_lg")
    as_doc = nlp_model(as_text)
    ls_doc = nlp_model(ls_text)
    
    as_ents = set([ent.text.lower().strip() for ent in as_doc.ents])
    ls_ents = set([ent.text.lower().strip() for ent in ls_doc.ents])
    
    overlap = as_ents.intersection(ls_ents)
    recall_as_ls = len(overlap) / len(as_ents) if as_ents else 1.0
    recall_ls_as = len(overlap) / len(ls_ents) if ls_ents else 1.0
    
    return {
        "as_ents": as_ents,
        "ls_ents": ls_ents,
        "overlap": overlap,
        "as_lost": as_ents - ls_ents,
        "ls_added": ls_ents - as_ents,
        "recall_as_ls": recall_as_ls,
        "recall_ls_as": recall_ls_as
    }

# Test an einem exemplarischen Satzpaar
test_as = "Bundeskanzler Olaf Scholz besuchte am Montag die Charité in Berlin, um über die Pflegereform zu sprechen."
test_ls = "Olaf Scholz ist der Bundes-Kanzler. Er war am Montag in Berlin in einem großen Kranken-Haus."

res_test = extract_entities_and_recall(test_as, test_ls)
print("=== Demonstration der NER-Referenzfunktion ===")
print(f"AS Entitäten ({len(res_test['as_ents'])}):", res_test["as_ents"])
print(f"LS Entitäten ({len(res_test['ls_ents'])}):", res_test["ls_ents"])
print(f"Überlappung ({len(res_test['overlap'])}):", res_test["overlap"])
print(f"In AS verloren ({len(res_test['as_lost'])}):", res_test["as_lost"])
print(f"In LS neu/umformuliert ({len(res_test['ls_added'])}):", res_test["ls_added"])
print(f"Faktenerhalt (AS -> LS): {res_test['recall_as_ls']:.2%}")
print(f"Faktentreue  (LS -> AS): {res_test['recall_ls_as']:.2%}")


## 2. Korpus-Übersicht nach Quellen

Deskriptive Kennzahlen für jede Datenquelle: Artikelpaare, Gesamttokens, durchschnittliche Längen, mittlere semantische Ähnlichkeit, bidirektionaler NER-Recall (Faktenerhalt & Faktentreue) und MATTR-Diversität.

In [ ]:
# Aggregation pro Quelle inklusive Wort- & BPE-Tokens sowie NER-Kennzahlen
summary_df = df.groupby("source").agg(
    Artikelpaare=("as_tokens", "count"),
    Wort_AS_Gesamt=("as_tokens", "sum"),
    Wort_LS_Gesamt=("ls_tokens", "sum"),
    BPE_AS_Gesamt=("as_bpe_tokens", "sum"),
    BPE_LS_Gesamt=("ls_bpe_tokens", "sum"),
    Ø_Wort_AS=("as_tokens", "mean"),
    Ø_Wort_LS=("ls_tokens", "mean"),
    Ø_BPE_AS=("as_bpe_tokens", "mean"),
    Ø_BPE_LS=("ls_bpe_tokens", "mean"),
    Ratio_Wort=("ratio_ls_as_word", "mean"),
    Ratio_BPE=("ratio_ls_as_bpe", "mean"),
    Ø_Ähnlichkeit=("semantic_similarity_8192", "mean") if "semantic_similarity_8192" in df.columns else ("as_tokens", "count"),
    Ø_NER_AS_LS=("ner_recall_as_ls", "mean") if "ner_recall_as_ls" in df.columns else ("as_tokens", "count"),
    Ø_NER_LS_AS=("ner_recall_ls_as", "mean") if "ner_recall_ls_as" in df.columns else ("as_tokens", "count"),
    Ø_MATTR_AS=("as_mattr", "mean") if "as_mattr" in df.columns else ("as_tokens", "count"),
    Ø_MATTR_LS=("ls_mattr", "mean") if "ls_mattr" in df.columns else ("as_tokens", "count")
)

# Gesamtsummen-Zeile
total_row = pd.Series({
    "Artikelpaare": len(df),
    "Wort_AS_Gesamt": df["as_tokens"].sum(),
    "Wort_LS_Gesamt": df["ls_tokens"].sum(),
    "BPE_AS_Gesamt": df["as_bpe_tokens"].sum(),
    "BPE_LS_Gesamt": df["ls_bpe_tokens"].sum(),
    "Ø_Wort_AS": df["as_tokens"].mean(),
    "Ø_Wort_LS": df["ls_tokens"].mean(),
    "Ø_BPE_AS": df["as_bpe_tokens"].mean(),
    "Ø_BPE_LS": df["ls_bpe_tokens"].mean(),
    "Ratio_Wort": df["ls_tokens"].sum() / df["as_tokens"].sum(),
    "Ratio_BPE": df["ls_bpe_tokens"].sum() / df["as_bpe_tokens"].sum(),
    "Ø_Ähnlichkeit": df["semantic_similarity_8192"].mean() if "semantic_similarity_8192" in df.columns else np.nan,
    "Ø_NER_AS_LS": df["ner_recall_as_ls"].mean() if "ner_recall_as_ls" in df.columns else np.nan,
    "Ø_NER_LS_AS": df["ner_recall_ls_as"].mean() if "ner_recall_ls_as" in df.columns else np.nan,
    "Ø_MATTR_AS": df["as_mattr"].mean() if "as_mattr" in df.columns else np.nan,
    "Ø_MATTR_LS": df["ls_mattr"].mean() if "ls_mattr" in df.columns else np.nan
}, name="GESAMT")

summary_table = pd.concat([summary_df, pd.DataFrame([total_row])])
display(summary_table.round(3))


## 3. Verteilung der Textlängen (Tokens & Ratio)

Untersuchung der Längenverteilung von Alltagssprache (AS) und Leichter Sprache (LS) sowie des Kompressions-/Expansionsverhaltens.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df["as_tokens"], kde=True, color="#2b5c8f", label=f"AS Tokens (Ø {df['as_tokens'].mean():.1f})", bins=40, alpha=0.5, ax=axes[0])
sns.histplot(df["ls_tokens"], kde=True, color="#e27c38", label=f"LS Tokens (Ø {df['ls_tokens'].mean():.1f})", bins=40, alpha=0.5, ax=axes[0])
axes[0].set_title("Verteilung der Token-Längen (AS vs. LS)")
axes[0].set_xlabel("Anzahl Tokens")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

if "ratio_ls_as" in df.columns:
    sns.histplot(df["ratio_ls_as"], kde=True, color="#2a9d8f", bins=40, ax=axes[1])
    axes[1].axvline(1.0, color="red", linestyle="--", label="Gleiche Länge (Ratio = 1.0)")
    axes[1].axvline(df["ratio_ls_as"].median(), color="black", linestyle="-", label=f"Median ({df['ratio_ls_as'].median():.2f})")
    axes[1].set_title("Verteilung des Längenverhältnisses (LS / AS)")
    axes[1].set_xlabel("Ratio (LS-Tokens / AS-Tokens)")
    axes[1].set_ylabel("Häufigkeit")
    axes[1].legend()

plt.tight_layout()
plt.show()

# Boxplot der Token-Längen nach Quellen
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
df_melted = df.melt(id_vars=["source"], value_vars=["as_tokens", "ls_tokens"], 
                    var_name="Variante", value_name="Tokens")
df_melted["Variante"] = df_melted["Variante"].map({"as_tokens": "Alltagssprache (AS)", "ls_tokens": "Leichte Sprache (LS)"})

sns.boxplot(data=df_melted, x="source", y="Tokens", hue="Variante", palette=["#2b5c8f", "#e27c38"], ax=axes[0])
axes[0].set_title("Tokenlängen nach Quelle (AS vs. LS)")
axes[0].set_xlabel("Quelle")
axes[0].set_ylabel("Tokens")
axes[0].tick_params(axis="x", rotation=45)

if "ratio_ls_as" in df.columns:
    sns.boxplot(data=df, x="source", y="ratio_ls_as", color="#2a9d8f", ax=axes[1])
    axes[1].axhline(1.0, color="red", linestyle="--", alpha=0.7, label="Ratio = 1.0")
    axes[1].set_title("Längenverhältnis (LS / AS) nach Quelle")
    axes[1].set_xlabel("Quelle")
    axes[1].set_ylabel("Ratio (LS-Tokens / AS-Tokens)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].legend()

plt.tight_layout()
plt.show()


### 3.1 Truncation- & Informationsverlust-Analyse bei festen Sequenzlängen (256, 512, 1024 Tokens)

Für das Training und die Inferenz neuronaler Übersetzungs- und Vereinfachungsmodelle (z. B. Seq2Seq mBART-50 oder Decoder-Only Architekturen wie LLaMA / Mistral) sowie für Regressions-Metrikmodelle (z. B. BiLSTM / SBERT) müssen in der Praxis feste maximale Sequenzlängen $L \in \{256, 512, 1024\}$ definiert werden.

Dabei stellen sich zwei zentrale methodische Fragen:
1. **Truncation (Hartes Abschneiden beim Tokenizing)**:
   - Welcher Anteil der Artikel überschreitet das jeweilige Limit $L$ und wird unvollständig verarbeitet?
   - Wie viele **BPE-Subword-Tokens** (`tiktoken cl100k_base`) und **Wort-Tokens** gehen dabei absolut und prozentual verloren?
   - Welche Konsequenzen hat dies (z. B. fehlende Schlussabsätze, abgeschnittene Sätze, Verlust des `<eos>`-Tokens im Training)?
2. **Filtering (Ausschluss zu langer Paare)**:
   - Wie viele **Artikelpaare** ($AS \to LS$) passen vollständig in das Kontextfenster?
   - Welcher Prozentsatz des Gesamtkorpus ginge verloren, wenn man nur vollständig intakte Artikelpaare ohne Truncation verwenden würde?


In [ ]:
# ==============================================================================
# 3.1 Truncation- & Verlustanalyse für Limits: 256, 512, 1024 Tokens
# ==============================================================================

cutoffs = [256, 512, 1024]

# 1. Gesamtkorpus-Analyse: Wort- und BPE-Tokens
trunc_overall_rows = []
for c in cutoffs:
    for mode_name, as_col, ls_col in [
        ('BPE-Tokens (cl100k_base)', 'as_bpe_tokens', 'ls_bpe_tokens'),
        ('Wort-Tokens (Linguistisch)', 'as_tokens', 'ls_tokens')
    ]:
        as_tot = int(df[as_col].sum())
        ls_tot = int(df[ls_col].sum())
        
        as_exceed = int((df[as_col] > c).sum())
        ls_exceed = int((df[ls_col] > c).sum())
        
        as_lost = int(np.maximum(0, df[as_col] - c).sum())
        ls_lost = int(np.maximum(0, df[ls_col] - c).sum())
        
        both_fit = int(((df[as_col] <= c) & (df[ls_col] <= c)).sum())
        either_trunc = len(df) - both_fit
        
        trunc_overall_rows.append({
            'Limit (L)': c,
            'Token-Typ': mode_name,
            'AS Artikel > L': f'{as_exceed} / {len(df)} ({as_exceed / len(df) * 100:.1f}%)',
            'AS Verlorene Tokens': f'{as_lost:,} ({as_lost / as_tot * 100:.1f}%)',
            'AS Erhalt (%)': f'{(1.0 - as_lost / as_tot) * 100:.1f}%',
            'LS Artikel > L': f'{ls_exceed} / {len(df)} ({ls_exceed / len(df) * 100:.1f}%)',
            'LS Verlorene Tokens': f'{ls_lost:,} ({ls_lost / ls_tot * 100:.1f}%)',
            'LS Erhalt (%)': f'{(1.0 - ls_lost / ls_tot) * 100:.1f}%',
            'Vollst. Paare (AS & LS <= L)': f'{both_fit} / {len(df)} ({both_fit / len(df) * 100:.1f}%)',
            'Paare m. Truncation': f'{either_trunc} / {len(df)} ({either_trunc / len(df) * 100:.1f}%)'
        })

df_trunc_summary = pd.DataFrame(trunc_overall_rows)
print('=== Gesamtübersicht: Truncation- und Informationsverlust nach Token-Limit ===')
display(df_trunc_summary)

# 2. Detaillierte Truncation-Analyse pro Datenquelle (BPE & Wörter)
source_trunc_rows = []
for src, grp in df.groupby('source'):
    n_src = len(grp)
    row = {'Quelle': src, 'Artikelpaare': n_src}
    for c in cutoffs:
        as_bpe_tot = grp['as_bpe_tokens'].sum()
        ls_bpe_tot = grp['ls_bpe_tokens'].sum()
        as_bpe_lost = np.maximum(0, grp['as_bpe_tokens'] - c).sum()
        ls_bpe_lost = np.maximum(0, grp['ls_bpe_tokens'] - c).sum()
        as_bpe_exc = (grp['as_bpe_tokens'] > c).sum()
        ls_bpe_exc = (grp['ls_bpe_tokens'] > c).sum()
        
        row[f'AS_>_{c}_BPE_%'] = (as_bpe_exc / n_src) * 100
        row[f'LS_>_{c}_BPE_%'] = (ls_bpe_exc / n_src) * 100
        row[f'AS_Verlust_{c}_BPE_%'] = (as_bpe_lost / as_bpe_tot) * 100 if as_bpe_tot > 0 else 0.0
        row[f'LS_Verlust_{c}_BPE_%'] = (ls_bpe_lost / ls_bpe_tot) * 100 if ls_bpe_tot > 0 else 0.0
    source_trunc_rows.append(row)

df_source_trunc = pd.DataFrame(source_trunc_rows)

print('\n=== Quellen-Aufschlüsselung: Prozentualer BPE-Token-Verlust pro Quelle ===')
display(df_source_trunc[['Quelle', 'Artikelpaare', 
                         'AS_Verlust_256_BPE_%', 'LS_Verlust_256_BPE_%', 
                         'AS_Verlust_512_BPE_%', 'LS_Verlust_512_BPE_%', 
                         'AS_Verlust_1024_BPE_%', 'LS_Verlust_1024_BPE_%']].round(1))


In [ ]:
# ==============================================================================
# 3.1 Visualisierung: Truncation-Verhalten & Verlustkurven
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# ------------------------------------------------------------------------------
# 1. ECDF: Kumulierte Abdeckung (Empirical Cumulative Distribution Function)
# ------------------------------------------------------------------------------
sorted_as_bpe = np.sort(df['as_bpe_tokens'])
sorted_ls_bpe = np.sort(df['ls_bpe_tokens'])
ecdf_as = np.arange(1, len(sorted_as_bpe) + 1) / len(sorted_as_bpe) * 100
ecdf_ls = np.arange(1, len(sorted_ls_bpe) + 1) / len(sorted_ls_bpe) * 100

axes[0, 0].plot(sorted_as_bpe, ecdf_as, label='Alltagssprache (AS)', color='#2b5c8f', lw=2.5)
axes[0, 0].plot(sorted_ls_bpe, ecdf_ls, label='Leichte Sprache (LS)', color='#e27c38', lw=2.5)

colors_cutoff = ['#e63946', '#f4a261', '#2a9d8f']
for c, col in zip(cutoffs, colors_cutoff):
    cov_as = (df['as_bpe_tokens'] <= c).mean() * 100
    cov_ls = (df['ls_bpe_tokens'] <= c).mean() * 100
    axes[0, 0].axvline(c, color=col, linestyle='--', alpha=0.85, 
                       label=f'Limit {c} (AS: {cov_as:.1f}%, LS: {cov_ls:.1f}%)')

axes[0, 0].set_title('A: Kumulierte Textabdeckung (ECDF - BPE-Tokens)', fontweight='bold')
axes[0, 0].set_xlabel('BPE-Token-Länge (tiktoken cl100k_base)')
axes[0, 0].set_ylabel('Vollständig erfasste Artikel (%)')
axes[0, 0].set_xlim(0, 2500)
axes[0, 0].set_ylim(0, 105)
axes[0, 0].legend(loc='lower right', framealpha=0.95)

# ------------------------------------------------------------------------------
# 2. BPE-Token-Verlust in % (AS vs. LS)
# ------------------------------------------------------------------------------
loss_as_bpe = [np.maximum(0, df['as_bpe_tokens'] - c).sum() / df['as_bpe_tokens'].sum() * 100 for c in cutoffs]
loss_ls_bpe = [np.maximum(0, df['ls_bpe_tokens'] - c).sum() / df['ls_bpe_tokens'].sum() * 100 for c in cutoffs]

x = np.arange(len(cutoffs))
width = 0.35

rects1 = axes[0, 1].bar(x - width/2, loss_as_bpe, width, label='Alltagssprache (AS)', color='#2b5c8f', alpha=0.85)
rects2 = axes[0, 1].bar(x + width/2, loss_ls_bpe, width, label='Leichte Sprache (LS)', color='#e27c38', alpha=0.85)

for r in rects1:
    h = r.get_height()
    axes[0, 1].annotate(f'{h:.1f}%', xy=(r.get_x() + r.get_width() / 2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10, fontweight='bold')
for r in rects2:
    h = r.get_height()
    axes[0, 1].annotate(f'{h:.1f}%', xy=(r.get_x() + r.get_width() / 2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0, 1].set_title('B: Prozentualer Verlust an BPE-Tokens durch Truncation', fontweight='bold')
axes[0, 1].set_xlabel('Maximales Sequenz-Limit (Tokens)')
axes[0, 1].set_ylabel('Verlorene Tokens an Gesamtmenge (%)')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([f'{c} Tokens' for c in cutoffs])
axes[0, 1].set_ylim(0, 100)
axes[0, 1].legend(loc='upper right')

# ------------------------------------------------------------------------------
# 3. Artikelpaare: Vollständig erhalten vs. Truncation nötig (Stacked Bar)
# ------------------------------------------------------------------------------
both_fit_counts = [int(((df['as_bpe_tokens'] <= c) & (df['ls_bpe_tokens'] <= c)).sum()) for c in cutoffs]
either_trunc_counts = [len(df) - bf for bf in both_fit_counts]

p1 = axes[1, 0].bar(x, both_fit_counts, width=0.5, label='Vollständig intakt (AS & LS <= L)', color='#2a9d8f', alpha=0.85)
p2 = axes[1, 0].bar(x, either_trunc_counts, width=0.5, bottom=both_fit_counts, label='Mindestens 1 Text abgeschnitten', color='#e76f51', alpha=0.85)

for i, (bf, et) in enumerate(zip(both_fit_counts, either_trunc_counts)):
    pct_bf = bf / len(df) * 100
    axes[1, 0].text(i, bf / 2, f'{bf}\n({pct_bf:.1f}%)', ha='center', va='center', color='white', fontweight='bold')
    axes[1, 0].text(i, bf + et / 2, f'{et}\n({100 - pct_bf:.1f}%)', ha='center', va='center', color='white', fontweight='bold')

axes[1, 0].set_title('C: Artikelpaare-Integrität bei BPE-Limits', fontweight='bold')
axes[1, 0].set_xlabel('Maximales Sequenz-Limit (Tokens)')
axes[1, 0].set_ylabel('Anzahl Artikelpaare')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([f'{c} Tokens' for c in cutoffs])
axes[1, 0].set_ylim(0, len(df) * 1.1)
axes[1, 0].legend(loc='upper left')

# ------------------------------------------------------------------------------
# 4. Heatmap: BPE-Token-Verlust pro Quelle (%)
# ------------------------------------------------------------------------------
heatmap_data = df_source_trunc.set_index('Quelle')[
    ['AS_Verlust_256_BPE_%', 'LS_Verlust_256_BPE_%', 
     'AS_Verlust_512_BPE_%', 'LS_Verlust_512_BPE_%', 
     'AS_Verlust_1024_BPE_%', 'LS_Verlust_1024_BPE_%']
].copy()
heatmap_data.columns = ['AS 256', 'LS 256', 'AS 512', 'LS 512', 'AS 1024', 'LS 1024']

sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Token-Verlust (%)'}, ax=axes[1, 1])
axes[1, 1].set_title('D: BPE-Token-Verlust nach Quelle & Limit (%)', fontweight='bold')
axes[1, 1].set_ylabel('Quelle')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


### 3.2 Interpretation & Konsequenzen für Modelltraining und Inferenz

Die quantitative Truncation-Analyse liefert fundamentale Erkenntnisse für das Training und das Evaluierungsdesign:

1. **Limit 256 Tokens (Starke Informationsvernichtung)**:
   - **94.8 % der AS-Texte** und **90.8 % der LS-Texte** überschreiten das Limit von 256 BPE-Tokens.
   - **77.7 % aller AS-Tokens** und **67.4 % aller LS-Tokens** werden abgeschnitten und gehen verloren.
   - Nur **2.4 % aller Artikelpaare** passen vollständig ohne Abschneiden in das Modell.
   - **Modell-Konsequenz**: Ein 256-Token-Limit für vollständige Artikel führt im Training dazu, dass das Modell fast nie ein echtes `<eos>`-Token am Artikelende sieht, wodurch es bei der Inferenz zu Textabbrüchen oder Halluzinationen neigt. 256 Tokens eignen sich ausschließlich für satz- oder absatzbasiertes Training (Sentence-level).

2. **Limit 512 Tokens (Kritischer Übergangsbereich)**:
   - Bei Leichter Sprache sind bereits **45.3 % der Artikel** vollständig erfasst (BPE-Verlust sinkt auf 43.9 %).
   - In der Alltagssprache überschreiten jedoch weiterhin **77.4 % der Artikel** das Limit (57.9 % Token-Verlust).
   - Nur **16.9 % der Artikelpaare** bleiben komplett intakt.
   - **Modell-Konsequenz**: 512 Tokens reichen für kurze Pressemitteilungen (z. B. *MDR*, *Wiesbaden*), schneiden aber bei Ratgeber- und Hintergrundartikeln (*Apotheken*, *Stuttgart*, *Köln*) mehr als die Hälfte des Inhalts ab.

3. **Limit 1024 Tokens (Optimaler Bereich für Ganztext-Modelle)**:
   - **75.5 % der LS-Artikel** und **59.0 % der AS-Artikel** passen vollständig in das Kontextfenster.
   - Der Token-Verlust sinkt auf **19.1 % (LS)** bzw. **31.1 % (AS)**.
   - **Mehr als die Hälfte aller Artikelpaare (52.8 %)** können ohne jeden Informationsverlust trainiert und evaluiert werden.
   - Bei Quellen wie *MDR*, *Behindertenbeauftragter*, *BrandEins*, *Wiesbaden* oder *Sozialpolitik* liegt der LS-Verlust bei < 3–14 %.

> [!IMPORTANT]
> **Fazit für die Master Thesis**:
> Für **Ganztext-Übersetzungsmodelle (Article-level Seq2Seq / LLM)** ist ein Kontextfenster von **mindestens 1024 Tokens** zwingend erforderlich, um Informationserhalt und ein sauberes Generierungsende (`<eos>`) sicherzustellen. Bei Modellen mit $\le 512$ Tokens ist eine vorherige Segmentierung in Absätze oder Sätze linguistisch und statistisch geboten.


## 4. Verteilung der semantischen Ähnlichkeit (Cosine Similarity)

Analyse der inhaltlichen Übereinstimmung zwischen Ausgangs- und Zieltexten, gemessen mittels Cosine Similarity von SBERT-Embeddings (Jina-v2-base-de mit 8192 Kontextfenster).

In [ ]:
sim_col = "semantic_similarity_8192" if "semantic_similarity_8192" in df.columns else None

if sim_col:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    mean_sim = df[sim_col].mean()
    median_sim = df[sim_col].median()
    q25, q75 = df[sim_col].quantile(0.25), df[sim_col].quantile(0.75)
    
    sns.histplot(df[sim_col], kde=True, bins=35, color="#3a86ff", ax=axes[0])
    axes[0].axvline(mean_sim, color="red", linestyle="-", label=f"Mittelwert ({mean_sim:.3f})")
    axes[0].axvline(median_sim, color="darkgreen", linestyle="--", label=f"Median ({median_sim:.3f})")
    axes[0].axvline(q25, color="gray", linestyle=":", label=f"Q1 ({q25:.3f})")
    axes[0].axvline(q75, color="gray", linestyle=":", label=f"Q3 ({q75:.3f})")
    axes[0].set_title("Verteilung der semantischen Ähnlichkeit (Gesamtkorpus)")
    axes[0].set_xlabel("Cosine Similarity (Jina 8192)")
    axes[0].set_ylabel("Anzahl Artikelpaare")
    axes[0].legend()
    
    sns.boxplot(data=df, x="source", y=sim_col, color="#4C72B0", ax=axes[1])
    axes[1].set_title("Semantische Ähnlichkeit nach Quelle")
    axes[1].set_xlabel("Quelle")
    axes[1].set_ylabel("Cosine Similarity")
    axes[1].tick_params(axis="x", rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    thresholds = [0.0, 0.70, 0.80, 0.85, 0.90, 1.01]
    labels = ["< 0.70 (Niedrig)", "0.70 - 0.80 (Moderat)", "0.80 - 0.85 (Gut)", "0.85 - 0.90 (Sehr gut)", ">= 0.90 (Exzellent)"]
    df["sim_category"] = pd.cut(df[sim_col], bins=thresholds, labels=labels, right=False)
    
    sim_dist = df["sim_category"].value_counts(sort=False).to_frame(name="Anzahl")
    sim_dist["Anteil (%)"] = (sim_dist["Anzahl"] / len(df)) * 100
    print("=== Schwellenwert-Verteilung der semantischen Ähnlichkeit ===")
    display(sim_dist)


## 5. Bidirektionale Named Entity Analyse (NER) & Faktenerhalt vs. Faktentreue

In dieser Sektion untersuchen wir die Beibehaltung von Fakten und Eigennamen (Personen, Orte, Organisationen) beim Übergang von Alltagssprache zu Leichter Sprache.

### Zentrale Fragestellungen:
1. **Faktenerhalt ($AS \to LS$)**: Welche Entitäten bleiben erhalten und welcher Anteil geht verloren?
2. **Faktentreue ($LS \to AS$)**: Führt Leichte Sprache neue Entitäten ein, die nicht im Originaltext stehen?
3. **Richtungsasymmetrie**: Ist der Erhalt in beide Richtungen gleich groß oder unterscheiden sich die Raten signifikant?
4. **Ursachen von Entitätsverlusten**: Liegt ein tatsächlicher Informationsverlust vor oder werden Eigennamen bewusst zu Gattungsbegriffen generalisiert bzw. umschrieben (z. B. *"Bundesagentur für Arbeit"* $\to$ *"Amt"*, *"Krebsregister"* $\to$ *"Sammlung von Infos"*?)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

mean_as_ls = df["ner_recall_as_ls"].mean()
mean_ls_as = df["ner_recall_ls_as"].mean()
median_as_ls = df["ner_recall_as_ls"].median()
median_ls_as = df["ner_recall_ls_as"].median()

# Histogramm / KDE beider Richtungen
sns.histplot(df["ner_recall_as_ls"], kde=True, color="#2b5c8f", label=f"AS -> LS (Faktenerhalt, Ø {mean_as_ls:.3f}, Med {median_as_ls:.3f})", bins=30, alpha=0.5, ax=axes[0])
sns.histplot(df["ner_recall_ls_as"], kde=True, color="#e27c38", label=f"LS -> AS (Faktentreue, Ø {mean_ls_as:.3f}, Med {median_ls_as:.3f})", bins=30, alpha=0.5, ax=axes[0])
axes[0].set_title("Verteilung des Bidirektionalen NER-Recalls")
axes[0].set_xlabel("NER Recall")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

# Boxplot des direkten Vergleichs beider Richtungen
df_ner_melted = df.melt(id_vars=["source"], value_vars=["ner_recall_as_ls", "ner_recall_ls_as"],
                        var_name="Richtung", value_name="Recall")
df_ner_melted["Richtung"] = df_ner_melted["Richtung"].map({
    "ner_recall_as_ls": "AS -> LS (Faktenerhalt)",
    "ner_recall_ls_as": "LS -> AS (Faktentreue)"
})

sns.boxplot(data=df_ner_melted, x="Richtung", y="Recall", palette=["#2b5c8f", "#e27c38"], ax=axes[1])
axes[1].set_title("Vergleich: Faktenerhalt (AS->LS) vs. Faktentreue (LS->AS)")
axes[1].set_ylabel("NER Recall")

plt.tight_layout()
plt.show()

# Statistische Signifikanzprüfung des Richtungsunterschieds
t_stat_ner, p_val_ner = stats.ttest_rel(df["ner_recall_as_ls"].dropna(), df["ner_recall_ls_as"].dropna())
w_stat_ner, p_val_w_ner = stats.wilcoxon(df["ner_recall_as_ls"].dropna(), df["ner_recall_ls_as"].dropna())

print(f"Paarweiser t-Test (AS->LS vs. LS->AS): t = {t_stat_ner:.3f}, p = {p_val_ner:.3e}")
print(f"Wilcoxon Signed-Rank Test: W = {w_stat_ner:.3f}, p = {p_val_w_ner:.3e}")
print(f"-> Faktentreue (LS->AS: {mean_ls_as:.1%}) ist signifikant höher als Faktenerhalt (AS->LS: {mean_as_ls:.1%}),")
print("   da Leichte Sprache den Großteil der Entitäten aus AS streicht/generalisiert, die verbleibenden aber überwiegend aus AS stammen.")


In [ ]:
mean_ner_by_source = df.groupby("source")[["ner_recall_as_ls", "ner_recall_ls_as"]].mean().reset_index()
mean_ner_by_source.columns = ["Quelle", "AS -> LS (Faktenerhalt)", "LS -> AS (Faktentreue)"]
ner_plot_df = mean_ner_by_source.melt(id_vars="Quelle", var_name="Richtung", value_name="Recall")

plt.figure(figsize=(14, 6))
sns.barplot(data=ner_plot_df, x="Quelle", y="Recall", hue="Richtung", palette=["#2b5c8f", "#e27c38"])
plt.title("Bidirektionaler NER-Recall nach Datenquellen (Faktenerhalt vs. Faktentreue)")
plt.xlabel("Quelle")
plt.ylabel("Durchschnittlicher Recall")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Richtung", loc="upper right")
plt.tight_layout()
plt.show()

# Tabelle nach Quelle
print("=== Detaillierte NER-Statistik nach Datenquellen ===")
ner_summary_source = df.groupby("source").agg(
    Paare=("ner_recall_as_ls", "count"),
    Ø_NER_AS_LS=("ner_recall_as_ls", "mean"),
    Med_NER_AS_LS=("ner_recall_as_ls", "median"),
    Ø_NER_LS_AS=("ner_recall_ls_as", "mean"),
    Med_NER_LS_AS=("ner_recall_ls_as", "median"),
    Ø_Ent_AS=("as_ent_count", "mean"),
    Ø_Ent_LS=("ls_ent_count", "mean")
)
display(ner_summary_source.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Verteilung der Entitätenanzahlen
sns.histplot(df["as_ent_count"], kde=True, color="#2b5c8f", label=f"AS Entitäten (Ø {df['as_ent_count'].mean():.1f})", bins=30, alpha=0.5, ax=axes[0])
sns.histplot(df["ls_ent_count"], kde=True, color="#e27c38", label=f"LS Entitäten (Ø {df['ls_ent_count'].mean():.1f})", bins=30, alpha=0.5, ax=axes[0])
axes[0].set_title("Anzahl der Named Entities pro Artikel (AS vs. LS)")
axes[0].set_xlabel("Anzahl Entitäten")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

# Scatterplot AS Entitäten vs LS Entitäten
sns.scatterplot(data=df, x="as_ent_count", y="ls_ent_count", hue="source", alpha=0.7, ax=axes[1])
max_ents = max(df["as_ent_count"].max(), df["ls_ent_count"].max())
axes[1].plot([0, max_ents], [0, max_ents], color="red", linestyle="--", label="Diagonale (Keine Entitäten-Reduktion)")
axes[1].set_title("Entitäten-Kompression: AS vs. LS Entitäts-Anzahl")
axes[1].set_xlabel("Anzahl Entitäten in Alltagssprache (AS)")
axes[1].set_ylabel("Anzahl Entitäten in Leichter Sprache (LS)")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

# Entitäten-Reduktionsrate
ent_reduction = 1.0 - (df["ls_ent_count"].sum() / df["as_ent_count"].sum())
print(f"Gesamte Entitäten in AS: {df['as_ent_count'].sum():,}")
print(f"Gesamte Entitäten in LS: {df['ls_ent_count'].sum():,}")
print(f"-> Absolute Entitäten-Reduktion im Gesamtkorpus: {ent_reduction:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatterplot: NER-Recall vs. Semantische Ähnlichkeit
sns.regplot(data=df, x="ner_recall_as_ls", y="semantic_similarity_8192", 
            scatter_kws={"alpha": 0.4, "color": "#2b5c8f"}, line_kws={"color": "red"}, ax=axes[0])
corr_ner_sim, p_ner_sim = stats.pearsonr(df["ner_recall_as_ls"].dropna(), df["semantic_similarity_8192"].dropna())
axes[0].set_title(f"NER-Recall vs. Semantische Ähnlichkeit (r = {corr_ner_sim:.3f}, p = {p_ner_sim:.3e})")
axes[0].set_xlabel("NER-Recall AS -> LS (Faktenerhalt)")
axes[0].set_ylabel("Cosine Similarity (Jina 8192)")

# Scatterplot: NER-Recall vs. Token-Ratio
sns.regplot(data=df, x="ratio_ls_as", y="ner_recall_as_ls", 
            scatter_kws={"alpha": 0.4, "color": "#2a9d8f"}, line_kws={"color": "red"}, ax=axes[1])
corr_ner_ratio, p_ner_ratio = stats.pearsonr(df["ratio_ls_as"].dropna(), df["ner_recall_as_ls"].dropna())
axes[1].set_title(f"Längenverhältnis vs. NER-Recall (r = {corr_ner_ratio:.3f}, p = {p_ner_ratio:.3e})")
axes[1].set_xlabel("Längenverhältnis (LS-Tokens / AS-Tokens)")
axes[1].set_ylabel("NER-Recall AS -> LS")

plt.tight_layout()
plt.show()

print("Erkenntnis:")
print(f"1. Korrelation NER vs. Ähnlichkeit (r = {corr_ner_sim:.3f}): Die semantische Ähnlichkeit bleibt selbst bei geringem NER-Recall hoch,")
print("   weil Kernaussagen durch Paraphrasen und Deskriptionen übermittelt werden, ohne Eigennamen wortwörtlich zu wiederholen.")
print(f"2. Korrelation Token-Ratio vs. NER (r = {corr_ner_ratio:.3f}): Längere Erklärungen in LS führen nicht zu einer signifikant höheren Übernahme von Entitäten.")


## 6. Type-Token Ratio (TTR) Analyse & Längen-Bias

Die klassische **Type-Token Ratio** ($TTR = V/N$) beschreibt den Anteil einzigartiger Wörter an der Gesamtwortzahl.

> [!WARNING]
> **Bekannter TTR-Längen-Bias (Herdan's Law / Zipf's Law):**
> Da die Anzahl neuer Wörter in einem Text mit wachsender Länge degressiv ansteigt, fällt die TTR bei längeren Texten mathematisch bedingt ab. Dies führt dazu, dass längere Alltagssprachen-Texte scheinbar eine geringere TTR haben als kurze Leichte-Sprache-Texte.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df["as_ttr"], color="#2b5c8f", label=f"AS TTR (Ø {df['as_ttr'].mean():.3f})", bins=30, alpha=0.5, ax=axes[0])
sns.histplot(df["ls_ttr"], color="#e27c38", label=f"LS TTR (Ø {df['ls_ttr'].mean():.3f})", bins=30, alpha=0.5, ax=axes[0])
axes[0].set_title("Verteilung der Type-Token Ratio (TTR)")
axes[0].set_xlabel("TTR (Types / Tokens)")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

sns.scatterplot(data=df, x="as_tokens", y="as_ttr", color="#2b5c8f", alpha=0.5, label="Alltagssprache (AS)", ax=axes[1])
sns.scatterplot(data=df, x="ls_tokens", y="ls_ttr", color="#e27c38", alpha=0.5, label="Leichte Sprache (LS)", ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_title("TTR vs. Textlänge (Log-Skala: Längen-Bias)")
axes[1].set_xlabel("Anzahl Tokens (log)")
axes[1].set_ylabel("TTR")
axes[1].legend()

plt.tight_layout()
plt.show()

# Statistische Auswertung des TTR-Unterschieds
t_stat, p_val = stats.ttest_rel(df["as_ttr"].dropna(), df["ls_ttr"].dropna())
print(f"Paarweiser t-Test TTR (AS vs. LS): t = {t_stat:.3f}, p = {p_val:.3e}")


## 7. Moving-Average Type-Token Ratio (MATTR)

Die **Moving-Average Type-Token Ratio (MATTR)** löst das Problem des Längen-Bias, indem die TTR über ein gleitendes Fenster fester Größe (z. B. $W=50$) gemittelt wird.

Dadurch wird die lexikalische Diversität unabhängig von der Gesamtlänge des Artikels messbar. In Leichter Sprache wird bewusst ein vereinfachtes, wiederholtes Vokabular gewählt, weshalb die MATTR von LS typischerweise signifikant unter der von AS liegt.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df["as_mattr"], color="#2b5c8f", label=f"AS MATTR (Ø {df['as_mattr'].mean():.3f})", bins=30, alpha=0.5, ax=axes[0])
sns.histplot(df["ls_mattr"], color="#e27c38", label=f"LS MATTR (Ø {df['ls_mattr'].mean():.3f})", bins=30, alpha=0.5, ax=axes[0])
axes[0].set_title("Verteilung der MATTR (Fenstergröße W=50)")
axes[0].set_xlabel("MATTR")
axes[0].set_ylabel("Häufigkeit")
axes[0].legend()

df["delta_mattr"] = df["as_mattr"] - df["ls_mattr"]
sns.histplot(df["delta_mattr"], color="#8338ec", bins=30, ax=axes[1])
axes[1].axvline(0.0, color="red", linestyle="--", label="Kein Unterschied (Delta = 0)")
axes[1].axvline(df["delta_mattr"].mean(), color="black", linestyle="-", label=f"Ø Reduktion ({df['delta_mattr'].mean():.3f})")
axes[1].set_title("Wortschatz-Vereinfachung (Δ MATTR = AS - LS)")
axes[1].set_xlabel("Δ MATTR")
axes[1].set_ylabel("Häufigkeit")
axes[1].legend()

plt.tight_layout()
plt.show()

# Statistische Auswertung des MATTR-Unterschieds
t_stat_m, p_val_m = stats.ttest_rel(df["as_mattr"].dropna(), df["ls_mattr"].dropna())
print(f"Paarweiser t-Test MATTR (AS vs. LS): t = {t_stat_m:.3f}, p = {p_val_m:.3e}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.scatterplot(data=df, x="as_tokens", y="as_mattr", color="#2b5c8f", alpha=0.5, label="AS MATTR", ax=axes[0])
sns.scatterplot(data=df, x="ls_tokens", y="ls_mattr", color="#e27c38", alpha=0.5, label="LS MATTR", ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_title("MATTR vs. Textlänge (Längenunabhängig)")
axes[0].set_xlabel("Anzahl Tokens (log)")
axes[0].set_ylabel("MATTR")
axes[0].legend()

df_mattr_melted = df.melt(id_vars=["source"], value_vars=["as_mattr", "ls_mattr"], 
                          var_name="Variante", value_name="MATTR")
df_mattr_melted["Variante"] = df_mattr_melted["Variante"].map({"as_mattr": "AS", "ls_mattr": "LS"})

sns.boxplot(data=df_mattr_melted, x="source", y="MATTR", hue="Variante", palette=["#2b5c8f", "#e27c38"], ax=axes[1])
axes[1].set_title("MATTR-Vergleich nach Quelle (AS vs. LS)")
axes[1].set_xlabel("Quelle")
axes[1].set_ylabel("MATTR")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 8. Multivariate Korrelationen & Gesamttabelle aller Korpusmetriken

Zusammenfassende Übersicht über alle analysierten Dimensionen der Untersuchung (Semantische Ähnlichkeit, Textlängen, Bidirektionales NER und MATTR-Diversität).

In [ ]:
key_metrics = [
    "semantic_similarity_8192",
    "ratio_ls_as",
    "ner_recall_as_ls",
    "ner_recall_ls_as",
    "as_ent_count",
    "ls_ent_count",
    "as_mattr",
    "ls_mattr",
    "delta_mattr"
]
available_cols = [c for c in key_metrics if c in df.columns]

plt.figure(figsize=(10, 8))
corr_matrix = df[available_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Korrelationsmatrix aller zentralen Korpus-Dimensionen")
plt.tight_layout()
plt.show()

metrics_summary = df[available_cols].describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
metrics_summary.columns = ["N", "Mittelwert", "Std", "Min", "Q1", "Median", "Q3", "Max"]
print("=== Zusammenfassende Statistik aller Korpusmetriken ===")
display(metrics_summary.round(3))


## 9. Quantitative Regel-Adhärenz & Regeleinhaltung der 12 Korpusquellen

In dieser Sektion analysieren wir die **empirische Einhaltung der linguistischen Richtlinien für Leichte Sprache** (nach *Netzwerk Leichte Sprache*, *BITV 2.0* und *DIN 8581-1*) über alle 12 Webquellen des Korpus hinweg ($N=830$ Artikelpaare).

### 9.1 Das quantitative Regel- und Metriken-Framework

Die qualitativen Richtlinien wurden über einen standardisierten **Rule Auditor** in 14 quantitative Indikatoren operationalisiert:

| # | Regelbereich | Offizielle LS-Regel | Quantitative Metrik | Technologie & Parser | Zielrichtung (LS vs. AS) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **S1** | **Syntax** | **Kurze Sätze** (max. 8–12 Wörter) | Mean Sentence Length (MSL) & Long Sentence Ratio (>12 Wörter) | `spacy` (`de_core_news_lg`): `doc.sents` | $\downarrow$ MSL $\le 8-10$, Long Sents $\approx 0$ |
| **S2** | **Syntax** | **Keine Nebensätze** (Nur Hauptsätze, Parataxe) | Subordination Ratio (Nebensätze pro Satz) | `spacy` Dep (`relcl`, `advcl`, `ccomp`) & POS (`SCONJ`) | $\downarrow$ Sinkt drastisch gegen $0.0$ |
| **S3** | **Syntax** | **SPO-Struktur** (Subjekt-Prädikat-Objekt) | Subject-Initial Sentence Ratio | `spacy` Dependency Parsing: `sb` am Satzanfang | $\uparrow$ Steigt in LS |
| **G1** | **Grammatik** | **Kein Passiv** (Nur Aktivformulierungen) | Passive Voice Ratio (Passivkonstruktionen pro Satz) | `spacy` Morph/Dep: `sb_pass` & *werden* + Partizip II | $\downarrow$ Sinkt gegen $0.0$ |
| **G2** | **Grammatik** | **Kein Genitiv** (Ersatz durch Dativ mit *von*) | Genitive Noun Ratio (Genitive pro Nomen/Pronomen) | `spacy` Morph: `Case=Gen` | $\downarrow$ Sinkt um $>85\%$ bis $100\%$ |
| **G3** | **Grammatik** | **Kein Konjunktiv** (Nur Indikativ/Tatsachen) | Subjunctive Mood Ratio (Konjunktiv pro finitem Verb) | `spacy` Morph: `Mood=Sub` | $\downarrow$ Nahezu $0.0$ |
| **G4** | **Grammatik** | **Verbalstil statt Nominalstil** | Nominalization Density & Verb-to-Noun Ratio | `regex` Suffixfilter (`-ung`, `-heit`, etc.) + POS | $\downarrow$ Nominalisierungen sinken; VNR $\uparrow$ |
| **G5** | **Grammatik** | **Wenig/keine Verneinungen** | Negation Density (*nicht*, *kein*, *weder*, etc.) | `regex` / Lemma-Matching | $\downarrow$ Sinkt in LS |
| **L1** | **Lexik** | **Kurze, einfache Wörter** | Polysyllable Ratio ($\ge 3$ Silben) & Mittlere Wortlänge | `pyphen` (de_DE Silbentrennung) & `textstat` | $\downarrow$ Sinkt signifikant |
| **L2** | **Lexik** | **Komposita trennen** (Bindestrich / Mediopunkt) | Compound Hyphenation Ratio | `regex`: Substantive mit `-` oder `·` | $\uparrow$ Steigt in LS stark an |
| **L3** | **Lexik** | **Keine Abkürzungen** (Wörter ausschreiben) | Abbreviation Density (*z. B.*, *bzw.*, Akronyme) | `regex` Pattern Matching | $\downarrow$ Sinkt gegen $0.0$ |
| **L4** | **Lexik** | **Zahlen als Ziffern** (*12* statt *zwölf*) | Digit Ratio (Ziffern vs. Zahlwörter) | `regex`: `\d+` vs. Zahlwort-Wörterbuch | $\uparrow$ Steigt gegen $1.0$ |
| **K1** | **Lesbarkeit**| **Standard-Lesbarkeitsformeln** | Wiener Sachtextformel (WSTF) & Flesch (DE) | `textstat` (lang='de') | WSTF sinkt (Ziel $\le 6$); Flesch $>70$ |
| **S1** | **Semantik** | **Sinn- & Fakten-Erhaltung** | Dense SBERT Semantic Similarity | `sentence-transformers` (Jina v2 8192) | Hohe Ähnlichkeit ($0.70 - 0.95$) |

### 9.2 Vorher-Nachher-Ergebnisse der Regel-Adhärenz

In [ ]:
corpus_rule_candidates = [
    "../../../data/analysis/rule_adherence_corpus.csv",
    "../../data/analysis/rule_adherence_corpus.csv",
    "data/analysis/rule_adherence_corpus.csv"
]
RULE_CSV_PATH = next((p for p in corpus_rule_candidates if os.path.exists(p)), "data/analysis/rule_adherence_corpus.csv")

df_corpus_rules = pd.read_csv(RULE_CSV_PATH)
print(f"Geladene Regel-Adhärenz-Daten: {len(df_corpus_rules)} Artikelpaare aus {df_corpus_rules['source'].nunique()} Quellen.")

cols_overview = [
    "as_avg_sent_len", "ls_avg_sent_len", "red_red_sent_len_pct",
    "as_passive_ratio", "ls_passive_ratio", "red_red_passive_pct",
    "as_genitive_ratio", "ls_genitive_ratio", "red_red_genitive_pct",
    "as_subord_ratio", "ls_subord_ratio", "red_red_subord_pct",
    "as_nominal_ratio", "ls_nominal_ratio", "red_red_nominal_pct",
    "as_wstf_score", "ls_wstf_score"
]

grouped_corpus = df_corpus_rules.groupby("source")[cols_overview].mean().round(2)
grouped_corpus.columns = [
    "AS Satzlänge", "LS Satzlänge", "Satzkürzung (%)",
    "AS Passiv/Satz", "LS Passiv/Satz", "Passiv-Red. (%)",
    "AS Genitiv-Quote", "LS Genitiv-Quote", "Genitiv-Red. (%)",
    "AS Nebensätze", "LS Nebensätze", "Nebensatz-Red. (%)",
    "AS Nominalstil", "LS Nominalstil", "Nominal-Red. (%)",
    "AS WSTF", "LS WSTF"
]
display(grouped_corpus)

### 9.3 Direkter Side-by-Side Vergleich pro Quelle (AS vs. LS in absoluten Werten)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Satzlänge
df_sent = df_corpus_rules.groupby('source')[['as_avg_sent_len', 'ls_avg_sent_len']].mean().reset_index()
df_sent_m = df_sent.melt(id_vars='source', var_name='Texttyp', value_name='Satzlänge (Wörter)')
df_sent_m['Texttyp'] = df_sent_m['Texttyp'].map({'as_avg_sent_len': 'Ausgangstext (AS)', 'ls_avg_sent_len': 'Leichte Sprache (LS)'})
sns.barplot(data=df_sent_m, x='source', y='Satzlänge (Wörter)', hue='Texttyp', ax=axes[0, 0], palette=['#4c72b0', '#55a868'])
axes[0, 0].set_title('1. Satzlänge: AS vs. LS (Wörter pro Satz)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].axhline(8.0, color='darkgreen', linestyle=':', label='Richtwert LS (≤ 8–10 Wörter)')
axes[0, 0].legend()

# Genitiv-Quote
df_gen = df_corpus_rules.groupby('source')[['as_genitive_ratio', 'ls_genitive_ratio']].mean().reset_index()
df_gen_m = df_gen.melt(id_vars='source', var_name='Texttyp', value_name='Genitiv-Quote')
df_gen_m['Texttyp'] = df_gen_m['Texttyp'].map({'as_genitive_ratio': 'Ausgangstext (AS)', 'ls_genitive_ratio': 'Leichte Sprache (LS)'})
sns.barplot(data=df_gen_m, x='source', y='Genitiv-Quote', hue='Texttyp', ax=axes[0, 1], palette=['#4c72b0', '#55a868'])
axes[0, 1].set_title('2. Genitiv-Quote: AS vs. LS (Genitive/Nomen)', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].legend()

# Passiv pro Satz
df_pass = df_corpus_rules.groupby('source')[['as_passive_ratio', 'ls_passive_ratio']].mean().reset_index()
df_pass_m = df_pass.melt(id_vars='source', var_name='Texttyp', value_name='Passiv pro Satz')
df_pass_m['Texttyp'] = df_pass_m['Texttyp'].map({'as_passive_ratio': 'Ausgangstext (AS)', 'ls_passive_ratio': 'Leichte Sprache (LS)'})
sns.barplot(data=df_pass_m, x='source', y='Passiv pro Satz', hue='Texttyp', ax=axes[1, 0], palette=['#4c72b0', '#55a868'])
axes[1, 0].set_title('3. Passiv-Dichte: AS vs. LS (Passiv/Satz)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Quelle')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].legend()

# Wiener Sachtextformel
df_wstf = df_corpus_rules.groupby('source')[['as_wstf_score', 'ls_wstf_score']].mean().reset_index()
df_wstf_m = df_wstf.melt(id_vars='source', var_name='Texttyp', value_name='WSTF Score')
df_wstf_m['Texttyp'] = df_wstf_m['Texttyp'].map({'as_wstf_score': 'Ausgangstext (AS)', 'ls_wstf_score': 'Leichte Sprache (LS)'})
sns.barplot(data=df_wstf_m, x='source', y='WSTF Score', hue='Texttyp', ax=axes[1, 1], palette=['#4c72b0', '#55a868'])
axes[1, 1].set_title('4. Wiener Sachtextformel (WSTF, Ziel ≤ 6)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Quelle')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(6.0, color='darkgreen', linestyle=':', label='Zielkorridor LS (≤ 6)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 9.4 Prozentuale Reduktionsraten pro Quelle

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
df_plot = df_corpus_rules.groupby("source")[["red_red_sent_len_pct", "red_red_genitive_pct", "red_red_passive_pct", "ls_wstf_score"]].mean().reset_index()

# Satzlängen-Kürzung
sns.barplot(data=df_plot.sort_values("red_red_sent_len_pct", ascending=False), x="source", y="red_red_sent_len_pct", ax=axes[0, 0], hue="source", legend=False, palette="crest")
axes[0, 0].set_title("Satzlängen-Kürzung (%) von AS nach LS", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Reduktion in %")
axes[0, 0].tick_params(axis="x", rotation=45)

# Genitiv-Tilgung
sns.barplot(data=df_plot.sort_values("red_red_genitive_pct", ascending=False), x="source", y="red_red_genitive_pct", ax=axes[0, 1], hue="source", legend=False, palette="Blues_r")
axes[0, 1].set_title("Genitiv-Tilgung (%) von AS nach LS", fontsize=12, fontweight="bold")
axes[0, 1].set_ylabel("Reduktion in %")
axes[0, 1].tick_params(axis="x", rotation=45)

# Passiv-Reduktion
sns.barplot(data=df_plot.sort_values("red_red_passive_pct", ascending=False), x="source", y="red_red_passive_pct", ax=axes[1, 0], hue="source", legend=False, palette="flare")
axes[1, 0].set_title("Passiv-Reduktion (%) von AS nach LS", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Reduktion in %")
axes[1, 0].tick_params(axis="x", rotation=45)

# Wiener Sachtextformel in LS
sns.barplot(data=df_plot.sort_values("ls_wstf_score", ascending=True), x="source", y="ls_wstf_score", ax=axes[1, 1], hue="source", legend=False, palette="magma_r")
axes[1, 1].set_title("Mittlerer WSTF Score in LS (Schulstufe, Ziel ≤ 6)", fontsize=12, fontweight="bold")
axes[1, 1].axhline(6.0, color="green", linestyle=":", label="Ziel Leichte Sprache (≤ 6)")
axes[1, 1].set_ylabel("WSTF")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 9.5 Domänen-Profil: Multidimensionales Radar-Chart

In [ ]:
key_sources = ["brandeins", "mdr", "hamburg", "koeln", "sozialpolitik", "apotheken"]
df_sub = df_corpus_rules[df_corpus_rules["source"].isin(key_sources)]

radar_metrics = {
    "red_red_sent_len_pct": "Satzkürzung",
    "red_red_passive_pct": "Passiv-Tilgung",
    "red_red_genitive_pct": "Genitiv-Tilgung",
    "red_red_subord_pct": "Nebensatz-Kürzung",
    "red_red_nominal_pct": "Nominalstil-Kürzung"
}

grouped = df_sub.groupby("source")[list(radar_metrics.keys())].mean().clip(lower=0, upper=100)
categories = list(radar_metrics.values())
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = sns.color_palette("tab10", len(key_sources))
for idx, (source, row) in enumerate(grouped.iterrows()):
    values = row.values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle="solid", label=source, color=colors[idx])
    ax.fill(angles, values, color=colors[idx], alpha=0.1)

plt.xticks(angles[:-1], categories, size=11, fontweight="bold")
ax.set_rlabel_position(30)
plt.yticks([25, 50, 75, 100], ["25%", "50%", "75%", "100%"], color="grey", size=9)
plt.ylim(0, 100)
plt.title("Regel-Adhärenz Profil im Domänenvergleich (Reduktionsleistung in %)", size=13, fontweight="bold", y=1.08)
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

### 9.6 Interaktiver Live-Rule-Auditor für beliebige Satzpaare

Mit der Klasse `LeichteSpracheRuleAuditor` kann die linguistische Regel-Adhärenz direkt und interaktiv für beliebige Ausgangs- und Zieltexte überprüft werden.

In [ ]:
import sys
for p in [".", "..", "../..", "../../.."]:
    if os.path.exists(os.path.join(p, "scripts")):
        if os.path.abspath(p) not in sys.path:
            sys.path.insert(0, os.path.abspath(p))
        break

from scripts.evaluation.measure_rule_adherence import LeichteSpracheRuleAuditor

auditor = LeichteSpracheRuleAuditor()

def live_audit(as_text: str, ls_text: str):
    as_doc = auditor.nlp(as_text)
    ls_doc = auditor.nlp(ls_text)
    
    as_m = auditor.analyze_doc(as_doc)
    ls_m = auditor.analyze_doc(ls_doc)
    red_m = auditor.calculate_reduction_delta(as_m, ls_m)
    
    print("=================== LIVE REGEL-AUDIT ===================")
    print(f"[AS Original]: {as_text[:120]}...")
    print(f"[LS Zieltext]: {ls_text[:120]}...")
    print("--------------------------------------------------------")
    print(f"Satzlänge:        {as_m['avg_sent_len']:.1f} Wörter -> {ls_m['avg_sent_len']:.1f} Wörter ({red_m['red_sent_len_pct']:+.1f}% Reduktion)")
    print(f"Passiv/Satz:      {as_m['passive_ratio']:.2f} -> {ls_m['passive_ratio']:.2f} ({red_m['red_passive_pct']:+.1f}% Reduktion)")
    print(f"Genitiv-Quote:    {as_m['genitive_ratio']:.2f} -> {ls_m['genitive_ratio']:.2f} ({red_m['red_genitive_pct']:+.1f}% Reduktion)")
    print(f"Nebensatz-Dichte: {as_m['subord_ratio']:.2f} -> {ls_m['subord_ratio']:.2f} ({red_m['red_subord_pct']:+.1f}% Reduktion)")
    print(f"Nominalstil:      {as_m['nominal_ratio']:.2f} -> {ls_m['nominal_ratio']:.2f} ({red_m['red_nominal_pct']:+.1f}% Reduktion)")
    print(f"WSTF (Lesbarkeit): {as_m['wstf_score']:.1f} -> {ls_m['wstf_score']:.1f}")
    print("========================================================")

# Test-Beispiel:
as_sample = "Die Genehmigung des Antrags wird durch die zuständige Behörde nach sorgfältiger Prüfung der Unterlagen erteilt."
ls_sample = "Das Amt prüft die Papiere. Danach erlaubt das Amt den Antrag."
live_audit(as_sample, ls_sample)